# Raw SVI diagnostics

Fit one Raw SVI slice per expiry to the screened SPX midpoint-IV panel. This notebook summarizes calibration, visualizes representative smiles and the fitted surface, and checks residuals, held-out strikes, calibration stability, butterfly conditions, and calendar monotonicity. The checks are diagnostics; sampled grids do not prove absence of arbitrage.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.raw_svi import (
    fit_raw_svi_surface,
    raw_svi_butterfly_diagnostic,
    raw_svi_total_variance,
)

PANEL_PATH = ROOT / 'data/processed/spx_iv_panel_2025-08-29.csv'
panel = pd.read_csv(PANEL_PATH, parse_dates=['quote_date', 'expiry_date'])
result = fit_raw_svi_surface(panel)
observations = result['observations'].copy()
parameters = result['parameters_by_expiry'].copy()
observations['days_to_expiry'] = (observations['expiry_date'] - observations['quote_date']).dt.days
observations['fitted_iv'] = np.sqrt(observations['fitted_w'] / observations['time_to_expiry'])
observations['iv_residual'] = observations['fitted_iv'] - observations['mid_iv']
print(f'Input panel: {len(panel):,} rows; fitted OTM quotes: {len(observations):,}; expiries: {len(parameters)}')
print(f'Arbitrage-aware calibration: {result["arbitrage_constrained"]}')

## Fit summary

Each expiry is calibrated independently by least squares in total variance, $w=\sigma^2\tau$. The table reports the five Raw SVI parameters and in-sample RMSE. These unconstrained fits are not automatically arbitrage-free.

In [ ]:
fit_summary = parameters.merge(
    observations.groupby('expiry_date').agg(
        days_to_expiry=('days_to_expiry', 'first'),
        n_quotes=('market_w', 'size'),
        k_min=('log_moneyness', 'min'),
        k_max=('log_moneyness', 'max'),
    ).reset_index(),
    on='expiry_date',
)
fit_summary = fit_summary[['expiry_date', 'days_to_expiry', 'n_quotes', 'a', 'b', 'rho', 'm', 'sigma', 'rmse', 'k_min', 'k_max']]
display(fit_summary.assign(expiry_date=fit_summary['expiry_date'].dt.strftime('%Y-%m-%d')).round(6))
display(fit_summary[['n_quotes', 'rmse']].describe().round(6))

## Representative 2D smiles

Select the available expiries nearest 7, 30, 90, and 300 DTE. Points are screened market observations; each line is that expiry's fitted Raw SVI total variance curve. The line is shown only over the observed $k$ range, so this view does not disguise extrapolation as market fit.

In [ ]:
expiry_info = observations[['expiry_date', 'days_to_expiry']].drop_duplicates().sort_values('days_to_expiry')
selected_expiries = []
for target in [7, 30, 90, 300]:
    selected = expiry_info.iloc[(expiry_info['days_to_expiry'] - target).abs().argmin()]
    if selected['expiry_date'] not in [row['expiry_date'] for row in selected_expiries]:
        selected_expiries.append({'target_dte': target, **selected.to_dict()})

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
for ax, selected in zip(axes.flat, selected_expiries):
    expiry = selected['expiry_date']
    rows = observations.loc[observations['expiry_date'].eq(expiry)]
    params = parameters.loc[parameters['expiry_date'].eq(expiry)].iloc[0]
    k_grid = np.linspace(rows['log_moneyness'].min(), rows['log_moneyness'].max(), 400)
    p = [params[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    ax.scatter(rows['log_moneyness'], rows['market_w'], s=13, color='black', alpha=0.65, label='Observed')
    ax.plot(k_grid, raw_svi_total_variance(k_grid, *p), color='tab:blue', lw=2, label='Raw SVI fit')
    ax.set(title=f'{expiry.date()} ({selected["days_to_expiry"]} DTE; target {selected["target_dte"]})', xlabel='Log-moneyness k', ylabel='Total variance w')
    ax.grid(alpha=0.25)
    ax.legend()
for ax in axes.flat[len(selected_expiries):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## Observed and fitted 3D total-variance surface

Rotate the figure to inspect the screened observations and the fitted expiry slices together. The mesh is evaluated only within each expiry's observed $k$ range; gaps between expiry ranges are left blank. The model is slice-by-slice Raw SVI, not a globally parameterized surface.

In [ ]:
k_mesh_grid = np.linspace(observations['log_moneyness'].min(), observations['log_moneyness'].max(), 180)
expiry_days = fit_summary['days_to_expiry'].to_numpy()
w_mesh = np.full((len(fit_summary), len(k_mesh_grid)), np.nan)
for i, params in fit_summary.iterrows():
    inside = (k_mesh_grid >= params['k_min']) & (k_mesh_grid <= params['k_max'])
    p = [params[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    w_mesh[i, inside] = raw_svi_total_variance(k_mesh_grid[inside], *p)

fig = go.Figure()
fig.add_trace(go.Surface(
    x=k_mesh_grid, y=expiry_days, z=w_mesh, colorscale='Viridis', opacity=0.68,
    connectgaps=False, name='Fitted Raw SVI', colorbar=dict(title='Fitted w'),
))
fig.add_trace(go.Scatter3d(
    x=observations['log_moneyness'], y=observations['days_to_expiry'], z=observations['market_w'],
    mode='markers', marker=dict(size=2, color='black', opacity=0.6), name='Observed w',
    customdata=np.column_stack([observations['fitted_w'], observations['residual_w']]),
    hovertemplate='k=%{x:.3f}<br>DTE=%{y:.0f}<br>Observed w=%{z:.5f}<br>Fitted w=%{customdata[0]:.5f}<br>Residual=%{customdata[1]:+.5f}<extra></extra>',
))
fig.update_layout(title='Observed quotes and fitted Raw SVI slices', width=1000, height=750, scene=dict(xaxis_title='Log-moneyness k', yaxis_title='Days to expiry', zaxis_title='Total variance w'))
fig.show()

## Residual diagnostics

Residuals are fitted minus observed total variance. Look for systematic patterns with moneyness, maturity, or fitted level; a low aggregate RMSE can conceal structured errors. The second plot expresses errors in IV points for easier market interpretation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(observations['log_moneyness'], observations['residual_w'], s=8, alpha=0.35, color='black')
axes[0].axhline(0, color='tab:red', ls='--')
axes[0].set(xlabel='Log-moneyness k', ylabel='Residual in w', title='Residual vs moneyness')
axes[1].scatter(observations['days_to_expiry'], observations['residual_w'], s=8, alpha=0.35, color='black')
axes[1].axhline(0, color='tab:red', ls='--')
axes[1].set(xlabel='Days to expiry', ylabel='Residual in w', title='Residual vs maturity')
axes[2].scatter(observations['mid_iv'], observations['iv_residual'], s=8, alpha=0.35, color='black')
axes[2].axhline(0, color='tab:red', ls='--')
axes[2].set(xlabel='Observed midpoint IV', ylabel='Residual in IV', title='IV residual vs observed IV')
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()
display(observations[['residual_w', 'iv_residual']].describe().round(6))

## Held-out-strike validation

Within each expiry, sort by $k$ and hold out every fifth quote. Fit on the remaining quotes and evaluate the withheld strikes. This tests interpolation across the observed smile; it does not test maturity extrapolation. Compare train and test RMSE per expiry to spot overfitting.

In [ ]:
holdout_rows = []
for expiry, rows in observations.groupby('expiry_date', sort=True):
    rows = rows.sort_values('log_moneyness').reset_index(drop=True)
    test_mask = np.arange(len(rows)) % 5 == 0
    train, test = rows.loc[~test_mask].copy(), rows.loc[test_mask].copy()
    fitted = fit_raw_svi_surface(train)['parameters_by_expiry'].iloc[0]
    p = [fitted[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    test_w = raw_svi_total_variance(test['log_moneyness'].to_numpy(), *p)
    holdout_rows.append({
        'expiry_date': expiry, 'n_train': len(train), 'n_test': len(test),
        'train_rmse_w': fitted['rmse'],
        'test_rmse_w': float(np.sqrt(np.mean((test_w - test['market_w'].to_numpy()) ** 2))),
    })
holdout = pd.DataFrame(holdout_rows)
display(holdout.assign(expiry_date=holdout['expiry_date'].dt.strftime('%Y-%m-%d')).round(7))
display(holdout[['train_rmse_w', 'test_rmse_w']].describe().round(7))

## Calibration stability

Refit representative expiries from deliberately different starting parameters. Compare both parameter estimates and the fitted curve over the observed strike range. Parameters can vary while the fitted curves remain close, indicating weak parameter identification rather than materially different fits.

In [ ]:
stability_rows = []
for selected in selected_expiries:
    expiry = selected['expiry_date']
    rows = observations.loc[observations['expiry_date'].eq(expiry)]
    k = rows['log_moneyness'].to_numpy()
    grid = np.linspace(k.min(), k.max(), 300)
    min_w = float(rows['market_w'].min())
    starts = {
        'default': None,
        'negative_skew': [max(min_w - 0.01, -1.0), 0.03, -0.8, 0.0, 0.05],
        'symmetric': [max(min_w - 0.01, -1.0), 0.15, 0.0, 0.0, 0.2],
        'positive_skew': [max(min_w - 0.01, -1.0), 0.3, 0.7, 0.0, 0.35],
    }
    reference = None
    for name, start in starts.items():
        try:
            params = fit_raw_svi_surface(rows, initial_guess=start)['parameters_by_expiry'].iloc[0]
            p = [params[col] for col in ['a', 'b', 'rho', 'm', 'sigma']]
            curve = raw_svi_total_variance(grid, *p)
            if reference is None: reference = curve
            stability_rows.append({
                'expiry_date': expiry, 'start': name, 'rmse_w': params['rmse'],
                'max_abs_curve_difference': float(np.max(np.abs(curve - reference))),
                **dict(zip(['a', 'b', 'rho', 'm', 'sigma'], p)),
            })
        except ValueError as error:
            stability_rows.append({'expiry_date': expiry, 'start': name, 'error': str(error)})
stability = pd.DataFrame(stability_rows)
display(stability.assign(expiry_date=stability['expiry_date'].dt.strftime('%Y-%m-%d')).round(6))

## Butterfly diagnostics

For each expiry, evaluate total-variance positivity, Durrleman's $g(k)$ on the quoted interval and on the wider $[-5,5]$ grid, plus both wing constraints $b(1+\rho)<2$ and $b(1-\rho)<2$. A violation only on the wide grid is an extrapolation warning; a violation within quoted strikes is more directly relevant. Grid checks are not a formal proof over all real $k$.

In [ ]:
butterfly_rows = []
wide_grid = np.linspace(-5.0, 5.0, 4001)
for _, params in parameters.iterrows():
    rows = observations.loc[observations['expiry_date'].eq(params['expiry_date'])]
    p = [params[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    quoted_grid = np.linspace(rows['log_moneyness'].min(), rows['log_moneyness'].max(), 1001)
    quoted = raw_svi_butterfly_diagnostic(*p, quoted_grid)
    wide = raw_svi_butterfly_diagnostic(*p, wide_grid)
    butterfly_rows.append({
        'expiry_date': params['expiry_date'], 'k_min': quoted_grid.min(), 'k_max': quoted_grid.max(),
        'min_g_quoted': quoted['minimum_g_on_grid'], 'min_g_wide': wide['minimum_g_on_grid'],
        'g_nonnegative_quoted': quoted['g_nonnegative_on_grid'],
        'g_nonnegative_wide': wide['g_nonnegative_on_grid'],
        'variance_nonnegative': wide['nonnegative_total_variance'],
        'left_wing_slope': wide['left_wing_slope'], 'left_wing_pass': wide['left_wing_condition'],
        'right_wing_slope': wide['right_wing_slope'], 'right_wing_pass': wide['right_wing_condition'],
    })
butterfly = pd.DataFrame(butterfly_rows)
display(butterfly.assign(expiry_date=butterfly['expiry_date'].dt.strftime('%Y-%m-%d')).round(7))
display(butterfly[['g_nonnegative_quoted', 'g_nonnegative_wide', 'variance_nonnegative', 'left_wing_pass', 'right_wing_pass']].mean())

## Calendar diagnostics

A basic calendar check compares total variance at fixed $k$ for adjacent expiries: longer-dated total variance should not be below shorter-dated total variance. This is checked on each adjacent pair's overlapping observed moneyness interval and on a wider common $k$ interval. The overlap check is the more market-relevant one; the wider check diagnoses model extrapolation. This sampled check is not a complete static-arbitrage proof.

In [ ]:
ordered = fit_summary.sort_values('days_to_expiry').reset_index(drop=True)
calendar_rows = []
for i in range(len(ordered) - 1):
    near, far = ordered.iloc[i], ordered.iloc[i + 1]
    overlap_lo = max(near['k_min'], far['k_min'])
    overlap_hi = min(near['k_max'], far['k_max'])
    if overlap_lo < overlap_hi:
        overlap_k = np.linspace(overlap_lo, overlap_hi, 501)
        near_p = [near[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
        far_p = [far[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
        diff_overlap = raw_svi_total_variance(overlap_k, *far_p) - raw_svi_total_variance(overlap_k, *near_p)
        n_violations = int((diff_overlap < -1e-10).sum())
        min_diff = float(diff_overlap.min())
    else:
        n_violations, min_diff = np.nan, np.nan
    wide_k = np.linspace(-0.5, 0.5, 1001)
    near_p = [near[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    far_p = [far[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    diff_wide = raw_svi_total_variance(wide_k, *far_p) - raw_svi_total_variance(wide_k, *near_p)
    calendar_rows.append({
        'near_expiry': near['expiry_date'], 'near_dte': near['days_to_expiry'],
        'far_expiry': far['expiry_date'], 'far_dte': far['days_to_expiry'],
        'overlap_k_min': overlap_lo, 'overlap_k_max': overlap_hi,
        'overlap_min_w_difference': min_diff, 'overlap_violation_points': n_violations,
        'wide_min_w_difference': float(diff_wide.min()),
        'wide_violation_points': int((diff_wide < -1e-10).sum()),
    })
calendar = pd.DataFrame(calendar_rows)
display(calendar.assign(near_expiry=calendar['near_expiry'].dt.strftime('%Y-%m-%d'), far_expiry=calendar['far_expiry'].dt.strftime('%Y-%m-%d')).round(7))